# Gumbel checkpoint-selection gate (`v2_exit_gumbel_on_a100`)

Scores the in-training plateau checkpoints (iters 20–55) plus the already-gated
`last` (44% mean) on the trusted scorer: `scripts/eval_fast.py` machinery —
side-alternated, paired seeds, 3 seed batches, n=60/batch (n=180/ckpt).

**Runtime: CPU-only is fine** (eval is collection-bound; no GPU needed) — pick a
high-vCPU runtime. ~35–45 min total on 12 workers.

**Ship rule:** a checkpoint replaces `last` only if it beats it on **every** seed
batch (same rule as the control gate — guards the ~+4–5% selection bias from
maxing over 7 checkpoints). If several qualify, prefer the broad plateau
(iters 20–35) over a single-point spike.


In [ ]:
#@title 1. Setup: Mount Drive, Clone Repo, Install Dependencies

import os, sys

from google.colab import drive
drive.mount('/content/drive')

DRIVE_OUTPUT = '/content/drive/MyDrive/orbit_wars_outputs'
for sub in ['', '/checkpoints', '/logs', '/submissions']:
    os.makedirs(DRIVE_OUTPUT + sub, exist_ok=True)

REPO_URL = 'https://github.com/oshbocker/orbit_wars.git'  # <-- EDIT if needed
REPO_DIR = '/content/orbit_wars'
if os.path.exists(REPO_DIR):
    print('Repo present, pulling latest...')
    os.system(f'cd {REPO_DIR} && git pull')
else:
    os.system(f'git clone {REPO_URL} {REPO_DIR}')

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

os.system('pip install -q --upgrade "kaggle-environments>=1.28.0" pyyaml tensorboard')

CPU_COUNT = os.cpu_count() or 8

# Silence kaggle_environments' noisy OpenSpiel INFO logs in THIS kernel
# (subprocess training output is filtered by grep on the launch line instead).
import logging
for _ln in ('kaggle_environments.envs.open_spiel_env.open_spiel_env',
            'kaggle_environments.core_harness'):
    logging.getLogger(_ln).disabled = True

print(f'\nDrive: {DRIVE_OUTPUT} | cwd: {os.getcwd()} | vCPUs: {CPU_COUNT}')


In [ ]:
#@title 2. Score the plateau (3 seed batches, side-alternated, paired)
from pathlib import Path
from v2.config import load_v2_config, v2_config_to_dict
from scripts.eval_fast import _eval_checkpoint

RUN          = 'v2_exit_gumbel_on_a100'  #@param {type:"string"}
ITERS        = ['20', '25', '30', '35', '50', '55', 'last']
GAMES        = 60                        #@param {type:"integer"} # per seed batch
SEED_BATCHES = [20000, 31000, 42000]
EVAL_WORKERS = CPU_COUNT

cfg_dict = v2_config_to_dict(load_v2_config('configs/v2_exit_gumbel.yaml'))
ckpt_root = Path(f'{DRIVE_OUTPUT}/checkpoints') / RUN

header = f'{"iter":>6} | ' + ' '.join(f'seed{s:>6}' for s in SEED_BATCHES) + f' | {"mean":>5}'
print(f'run={RUN}  games/seed={GAMES}  workers={EVAL_WORKERS}\n')
print(header); print('-' * len(header))
results = {}
for it in ITERS:
    name = 'ckpt_last.pt' if it == 'last' else f'ckpt_{int(it):06d}.pt'
    path = ckpt_root / name
    if not path.exists():
        print(f'{it:>6} | (missing {name})'); continue
    wrs = []
    for s in SEED_BATCHES:
        win, loss, tie = _eval_checkpoint(path, cfg_dict, GAMES, EVAL_WORKERS, 'apex', s)
        wrs.append(win / max(win + loss + tie, 1))
    results[it] = wrs
    print(f'{it:>6} | ' + ' '.join(f'{r:>9.0%}' for r in wrs) + f' | {sum(wrs)/len(wrs):>5.0%}')

# Ship rule: a checkpoint replaces 'last' only if it beats it on EVERY seed batch.
if 'last' in results:
    base = results['last']
    print('\nvs last (the already-gated champion candidate), per-seed delta:')
    for it, wrs in results.items():
        if it == 'last': continue
        d = [a - b for a, b in zip(wrs, base)]
        print(f'  iter {it:>4}: ' + ' '.join(f'{x:+.0%}' for x in d)
              + f'   mean {sum(d)/len(d):+.1%}'
              + ('   <- beats last on every seed' if all(x > 0 for x in d) else ''))
